# US x1.0 — Canonical baseline model

`us_x1_0` is the first named US87 research baseline. Status: `trade_ready=false`. This notebook binds the model contract, provider identity, complete development windows, frozen challenge and reproduction commands.

## Effective parameter identity correction

The historical candidate name contained `learning_rate=0.03` and `min_data_in_leaf=20`, but the XGBoost adapter did not consume those LightGBM-oriented calibration fields. The actual `effective_runtime_parameters` are XGBoost `rank:ndcg`, `tree_method=hist`, `grow_policy=lossguide`, `max_leaves=31`, `max_depth=0`, `learning_rate=0.05`, `seed=42`, seven gain bins and 200 boosting rounds.

Therefore the supported improvement is attributed to the `risk_controlled_momentum` feature group, seven relevance gains and 200 rounds—not to a 0.03 learning rate or minimum-leaf regularization. The recorded economics are unchanged.

In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
import yaml

def repo_root(start=Path.cwd()):
    for path in (start.resolve(), *start.resolve().parents):
        if (path / 'pyproject.toml').is_file() and (path / 'configs').is_dir():
            return path
    raise FileNotFoundError('Run inside Alpha Engine')

ROOT = repo_root()
CONFIG = ROOT / 'configs/models/us_x1_0.yaml'
config = yaml.safe_load(CONFIG.read_text(encoding='utf-8'))
assert config['display_name'] == 'US x1.0'
assert config['trade_ready'] is False
assert config['model']['learning_rate'] == 0.05
assert config['candidate_calibration_identity']['effective_xgb_mapping']['learning_rate'] == 'not_consumed_by_xgb_adapter'
config['model']

## Contract

- Universe: `us_selected_equities_v2`, 87 declared equities; static curated and survivorship-biased.
- Benchmark: QQQ, reference only.
- Provider identity: `4921168fee3afdcfb222568bed370a0273a5d8795ac57e4477184c1728384a5c`, cutoff 2026-07-31.
- Features: seven `risk_controlled_momentum` expressions.
- Label/economics: daily cross-sectional gain target; raw 10-session forward return.
- Portfolio: Top-15 equal weight, 10-session holding/rebalance, 20 bps cost.

In [ ]:
pd.DataFrame({'feature_expression': config['features']['expressions']})

In [ ]:
identity = config['candidate_calibration_identity']
pd.DataFrame([
    {'layer': 'declared_identity', 'parameter': 'n_gain_bins', 'value': identity['n_gain_bins']},
    {'layer': 'declared_identity', 'parameter': 'num_boost_round', 'value': identity['num_boost_round']},
    {'layer': 'legacy_not_consumed', 'parameter': 'num_leaves', 'value': identity['legacy_num_leaves_field']},
    {'layer': 'legacy_not_consumed', 'parameter': 'min_data_in_leaf', 'value': identity['legacy_min_data_in_leaf_field']},
    {'layer': 'legacy_not_consumed', 'parameter': 'learning_rate', 'value': identity['legacy_learning_rate_field']},
    *[{'layer': 'effective_runtime_parameters', 'parameter': k, 'value': v} for k, v in config['model'].items() if k not in {'runtime_source', 'calibration_mapping_source', 'parameter_identity_status'}],
])

## Complete backtest evidence

Development covers 2024H1–2025H2. The authoritative `compounded_relative_excess` formula is `(1 + strategy) / (1 + benchmark) - 1`. The 2026H1 challenge was evaluated once in workflow run `30733686862`, artifact `8828827295`, and cannot be reused for candidate selection.

In [ ]:
dev = config['backtest_evidence']['development']
calculated = (1 + dev['compounded_strategy_return']) / (1 + dev['compounded_benchmark_return']) - 1
assert abs(calculated - dev['compounded_relative_excess_return']) < 1e-10
pd.DataFrame(dev['windows'])

In [ ]:
pd.DataFrame([
    {'metric': 'compounded_strategy_return', 'value': dev['compounded_strategy_return']},
    {'metric': 'compounded_benchmark_return', 'value': dev['compounded_benchmark_return']},
    {'metric': 'compounded_relative_excess', 'value': dev['compounded_relative_excess_return']},
    {'metric': 'mean_icir', 'value': dev['mean_icir']},
    {'metric': 'mean_rank_ic', 'value': dev['mean_rank_ic']},
    {'metric': 'mean_top_bottom_spread', 'value': dev['mean_top_bottom_spread']},
    {'metric': 'positive_excess_windows', 'value': dev['positive_excess_windows']},
    {'metric': 'worst_drawdown', 'value': dev['worst_drawdown']},
])

In [ ]:
pd.DataFrame([config['backtest_evidence']['frozen_challenge']]).T.rename(columns={0: '2026H1_value'})

## Interpretation and limitations

US x1.0 delivered +114.62% compounded relative excess across development and +88.18 percentage points of simple excess in 2026H1. However, 2025H1 drew down 28.36%, 2025H2 contributed disproportionately, and recurring high-beta growth/semiconductor selections require concentration analysis. US x1.0 remains immutable; an effective experiment may propose US x1.1 only after cost, concentration, bootstrap and a new untouched challenge gate.

In [ ]:
VALIDATE = [sys.executable, str(ROOT / 'scripts/validate_model_x1_baselines.py')]
FULL_BACKTEST = ['uv', 'run', 'python', 'scripts/run_us_feature_quality_validation.py', '--spec', 'configs/research_paradigms/us_10d_xgb_optimization_frozen_v1.yaml', '--provider-uri', 'artifacts/selected_pool_price_refresh/us/data/providers/us', '--output-dir', 'artifacts/evidence/model_versions/us_x1_0']
print('Contract validation:', ' '.join(VALIDATE))
print('Full backtest:', ' '.join(FULL_BACKTEST))
RUN = False
if RUN:
    subprocess.run(VALIDATE, cwd=ROOT, check=True)